In [45]:
import os
import oracledb
import pandas as pd

In [46]:
# Configuración de conexión desde variables de entorno
ORACLE_HOST = os.getenv('ORACLE_HOST')
ORACLE_PORT = os.getenv('ORACLE_PORT')
ORACLE_SERVICE = os.getenv('ORACLE_SERVICE')
ORACLE_USER = os.getenv('ORACLE_USER')
ORACLE_PASSWORD = os.getenv('ORACLE_PASSWORD')

dsn = f"{ORACLE_HOST}:{ORACLE_PORT}/{ORACLE_SERVICE}"

In [47]:
conn = oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=dsn)
cursor = conn.cursor()
print("Conexión establecida con Oracle DB.")

Conexión establecida con Oracle DB.


In [48]:
def ejecutar_consulta(query):
    cursor.execute(query)
    resultado = cursor.fetchall()
    df = pd.DataFrame(resultado, columns=[desc[0] for desc in cursor.description])
    display(df)

In [49]:
print("="*80)
print("Analítica 1: Ingresos y Pedidos últimos 12 meses (excluyendo pedidos cancelados):")
print("="*80)
ejecutar_consulta("""
                SELECT 
                    TO_CHAR(ORDER_DATE, 'YYYY-MM') AS MES,
                    COUNT(ORDER_ID) AS TOTAL_PEDIDOS,
                    ROUND(SUM(TOTAL_AMOUNT), 2) AS INGRESOS_TOTALES,
                    ROUND(AVG(TOTAL_AMOUNT), 2) AS TICKET_MEDIO
                FROM ORDERS
                WHERE STATUS != 'CANCELADO'
                GROUP BY TO_CHAR(ORDER_DATE, 'YYYY-MM')
                ORDER BY MES DESC
                FETCH FIRST 12 ROWS ONLY
            """)

Analítica 1: Ingresos y Pedidos últimos 12 meses (excluyendo pedidos cancelados):


,MES,TOTAL_PEDIDOS,INGRESOS_TOTALES,TICKET_MEDIO
0,2026-08,275,524040.17,1905.60
1,2026-07,238,433899.80,1823.11
2,2026-06,166,312561.92,1882.90
3,2026-05,154,270720.16,1757.92
4,2026-04,142,243738.05,1716.47
5,2026-03,126,222359.40,1764.76
6,2026-02,107,209464.58,1957.61
7,2026-01,117,206696.46,1766.64
8,2025-12,80,139046.90,1738.09
9,2025-11,85,159177.55,1872.68


In [50]:
print("="*80)
print("Analítica 2: Top 5 Productos Más Vendidos y su Rentabilidad")
print("="*80)
ejecutar_consulta("""
                SELECT 
                    p.PRODUCT_ID,
                    p.NAME AS PRODUCTO,
                    c.NAME AS CATEGORIA,
                    SUM(oi.QUANTITY) AS UNIDADES_VENDIDAS,
                    ROUND(SUM((oi.UNIT_PRICE * oi.QUANTITY) - oi.DISCOUNT), 2) AS FACTURACION_TOTAL,
                    ROUND(SUM((oi.UNIT_PRICE - p.COST) * oi.QUANTITY - oi.DISCOUNT), 2) AS BENEFICIO_ESTIMADO
                FROM ORDER_ITEMS oi
                JOIN PRODUCTS p ON oi.PRODUCT_ID = p.PRODUCT_ID
                JOIN CATEGORIES c ON p.CATEGORY_ID = c.CATEGORY_ID
                JOIN ORDERS o ON oi.ORDER_ID = o.ORDER_ID
                WHERE o.STATUS != 'CANCELADO'
                GROUP BY p.PRODUCT_ID, p.NAME, c.NAME
                ORDER BY UNIDADES_VENDIDAS DESC
                FETCH FIRST 5 ROWS ONLY
            """)

Analítica 2: Top 5 Productos Más Vendidos y su Rentabilidad


,PRODUCT_ID,PRODUCTO,CATEGORIA,UNIDADES_VENDIDAS,FACTURACION_TOTAL,BENEFICIO_ESTIMADO
0,9,Cross-group multi-tasking flexibility,Belleza y Cuidado Personal,268,76035.95,12670.03
1,56,Distributed analyzing website,Libros y Papelería,266,87573.37,17865.41
2,53,Universal leadingedge analyzer,Libros y Papelería,246,72295.97,10166.21
3,31,Multi-channeled contextually-based hardware,Hogar y Cocina,238,61643.56,20583.80
4,37,Implemented even-keeled open architecture,Informática,233,50275.25,20059.81


In [51]:
print("="*80)
print("Analítica 3: Tiempos Medios de Preparación y Entrega por País")
print("="*80)
ejecutar_consulta("""
                SELECT 
                    SEND_COUNTRY AS PAIS,
                    COUNT(ORDER_ID) AS PEDIDOS_ENTREGADOS,
                    ROUND(AVG(SEND_DATE - ORDER_DATE), 2) AS DIAS_ENVIO_MEDIO,
                    ROUND(AVG(RECEIVE_DATE - SEND_DATE), 2) AS DIAS_TRANSITO_MEDIO,
                    ROUND(AVG(RECEIVE_DATE - ORDER_DATE), 2) AS DIAS_TOTAL_ENTREGA
                FROM ORDERS
                WHERE STATUS = 'ENTREGADO' 
                  AND SEND_DATE IS NOT NULL 
                  AND RECEIVE_DATE IS NOT NULL
                GROUP BY SEND_COUNTRY
                ORDER BY PEDIDOS_ENTREGADOS DESC
            """)


Analítica 3: Tiempos Medios de Preparación y Entrega por País


,PAIS,PEDIDOS_ENTREGADOS,DIAS_ENVIO_MEDIO,DIAS_TRANSITO_MEDIO,DIAS_TOTAL_ENTREGA
0,España,1392,2,3.04,5.04


In [52]:
print("="*80)
print("Analítica 4: Satisfacción de Cliente y Rating Medio por Categoría")
print("="*80)
ejecutar_consulta("""
                SELECT 
                    c.NAME AS CATEGORIA,
                    COUNT(r.REVIEW_ID) AS TOTAL_RESEÑAS,
                    ROUND(AVG(r.RATING), 2) AS RATING_PROMEDIO,
                    SUM(CASE WHEN r.RATING >= 4 THEN 1 ELSE 0 END) AS RESEÑAS_POSITIVAS,
                    SUM(CASE WHEN r.RATING <= 2 THEN 1 ELSE 0 END) AS RESEÑAS_NEGATIVAS
                FROM REVIEWS r
                JOIN PRODUCTS p ON r.PRODUCT_ID = p.PRODUCT_ID
                JOIN CATEGORIES c ON p.CATEGORY_ID = c.CATEGORY_ID
                GROUP BY c.NAME
                ORDER BY RATING_PROMEDIO DESC
                """)

Analítica 4: Satisfacción de Cliente y Rating Medio por Categoría


,CATEGORIA,TOTAL_RESEÑAS,RATING_PROMEDIO,RESEÑAS_POSITIVAS,RESEÑAS_NEGATIVAS
0,Deportes y Exterior,145,4.26,122,8
1,Informática,137,4.17,112,8
2,Ropa y Moda,87,4.10,67,6
3,Belleza y Cuidado Personal,187,4.04,141,13
4,Electrónica,179,4.03,132,18
5,Juegos y Juguetes,79,4.01,56,9
6,Hogar y Cocina,103,4.01,77,11
7,Libros y Papelería,309,3.97,228,36


In [53]:
print("="*80)
print("Analítica 5: Rendimiento de Adquisición de Clientes por Canal")
print("="*80)
ejecutar_consulta("""
                SELECT 
                    c.CANAL,
                    COUNT(DISTINCT c.CUSTOMER_ID) AS TOTAL_CLIENTES,
                    COUNT(DISTINCT o.ORDER_ID) AS TOTAL_PEDIDOS,
                    ROUND(SUM(o.TOTAL_AMOUNT), 2) AS FACTURACION_ACUMULADA,
                    ROUND(SUM(o.TOTAL_AMOUNT) / COUNT(DISTINCT c.CUSTOMER_ID), 2) AS VALOR_MEDIO_CLIENTE
                FROM CUSTOMERS c
                LEFT JOIN ORDERS o ON c.CUSTOMER_ID = o.CUSTOMER_ID AND o.STATUS != 'CANCELADO'
                GROUP BY c.CANAL
                ORDER BY FACTURACION_ACUMULADA DESC
            """)

Analítica 5: Rendimiento de Adquisición de Clientes por Canal


,CANAL,TOTAL_CLIENTES,TOTAL_PEDIDOS,FACTURACION_ACUMULADA,VALOR_MEDIO_CLIENTE
0,TIENDA_FISICA,102,422,736386.85,7219.48
1,WEB,106,393,725860.71,6847.74
2,APP_IOS,104,389,723110.51,6952.99
3,APP_ANDROID,96,359,675427.58,7035.70
4,PUBLICIDAD,92,344,646654.47,7028.85


In [54]:
if cursor:
    cursor.close()
if conn:
    conn.close()